# Q3D Capacitance Postprocess

Parse PF6FQ Q0-Q2 Q3D `capMatrix` exports through the thesis-local workflow module. The module reuses existing `core` post-processing and simulation APIs where they apply.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists() and (path / "core").exists()
)
STUDY_DIR = REPO_ROOT / "sandbox/thesis_pf6fq_external_coupling_analysis"
for import_path in (REPO_ROOT, STUDY_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from q3d_xy_external_coupling import (  # noqa: E402
    DEFAULT_QUBITS,
    capacitance_summary_rows,
    load_floating_xy_capacitances,
)

RAW_LAYOUT_DIR = REPO_ROOT / "data/raw/layout_simulation/PF6FQ"
RAW_OUTPUT_DIR = STUDY_DIR / "outputs/raw"
TABLE_OUTPUT_DIR = STUDY_DIR / "outputs/tables"
for output_dir in (RAW_OUTPUT_DIR, TABLE_OUTPUT_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)

QUBITS = list(DEFAULT_QUBITS)

print(f"Repository root: {REPO_ROOT}")
print(f"Raw Q3D source: {RAW_LAYOUT_DIR}")
print(f"Qubits: {QUBITS}")

Repository root: /Users/arfiligol/Github/superconducting-circuits-tutorial
Raw Q3D source: /Users/arfiligol/Github/superconducting-circuits-tutorial/data/raw/layout_simulation/PF6FQ
Qubits: ['Q0', 'Q1', 'Q2']


## Capacitance Summary

In [2]:
cap_df = pd.DataFrame(
    capacitance_summary_rows(
        RAW_LAYOUT_DIR,
        QUBITS,
        repo_root=REPO_ROOT,
    )
)

raw_path = RAW_OUTPUT_DIR / "q3d_capacitance_parameters.csv"
table_path = TABLE_OUTPUT_DIR / "thesis_q3d_capacitance_summary.csv"
cap_df.to_csv(raw_path, index=False)
cap_df.to_csv(table_path, index=False)

display(cap_df)
print(f"Wrote {raw_path}")
print(f"Wrote {table_path}")

,qubit,source_unit,c_g1_ff,c_g2_ff,c_q_ff,c_xy1_ff,c_xy2_ff,c_xy_ground_ff,alpha,beta,c_d_xy_ff,c_dd_ff,c_eff_q_ff,source_path
0,Q0,fF,102.490356,101.825117,58.120811,0.174218,0.745141,627.804342,0.500230,0.499770,0.285673,109.429509,109.340342,data/raw/layout_simulation/PF6FQ/Q0/Q0_XY_Q3D_...
1,Q1,pF,102.562057,102.071150,58.198201,0.141059,0.559423,637.076994,0.500177,0.499823,0.209305,109.531617,109.468862,data/raw/layout_simulation/PF6FQ/Q1/Q1_XY_Q3D_...
2,Q2,pF,102.220149,102.099096,58.093312,0.134045,0.462584,569.443022,0.499494,0.500506,0.163967,109.322228,109.277035,data/raw/layout_simulation/PF6FQ/Q2/Q2_XY_Q3D_...


Wrote /Users/arfiligol/Github/superconducting-circuits-tutorial/sandbox/thesis_pf6fq_external_coupling_analysis/outputs/raw/q3d_capacitance_parameters.csv
Wrote /Users/arfiligol/Github/superconducting-circuits-tutorial/sandbox/thesis_pf6fq_external_coupling_analysis/outputs/tables/thesis_q3d_capacitance_summary.csv


## Per-Qubit Derived Values

In [3]:
for qubit in QUBITS:
    cap = load_floating_xy_capacitances(RAW_LAYOUT_DIR, qubit)
    print(
        f"{qubit}: alpha={cap.alpha:.6f}, beta={cap.beta:.6f}, "
        f"Ceff,q={cap.c_eff_q_f / 1e-15:.6f} fF, "
        f"C_d,XY={cap.c_d_xy_f / 1e-15:.6f} fF"
    )

Q0: alpha=0.500230, beta=0.499770, Ceff,q=109.340342 fF, C_d,XY=0.285673 fF
Q1: alpha=0.500177, beta=0.499823, Ceff,q=109.468862 fF, C_d,XY=0.209305 fF
Q2: alpha=0.499494, beta=0.500506, Ceff,q=109.277035 fF, C_d,XY=0.163967 fF
